Tcond is not chosen, it is set by the cold sink temperature. T_cond = wet bulb + approach + range + pinch.

Wet bulb - obtained from weather data

Approach temp - choose constant 4 degrees from "Modelling the influence paper"

Range - chosen, Claude and Thermo-economic comparison of air-cooled and cooling tower recommend around 10 F = 5K range (but then should iterate to find optimum)

Condensor pinch point - 5K, from the table 1 in Thermo-economic comparison of air-cooled and cooling tower 

In [29]:
T_wb_amb = 10 + 273.15
T_PP_condensor = 5
T_approach = 4
T_range = 5
T_cond = T_wb_amb + T_approach + T_range + T_PP_condensor

T_evap = 30 + 273.15 #Set T_evap

print(orc(T_evap, T_cond))

{'m_wf': 15.647965818804678, 'Q_in': 3110572.9499580157, 'T_w_out_evap': 307.22121226049035, 'W_net': 48128.2817276693, 'Q_cond': 3062444.6682303464, 'OCR_efficiency': 1.5472481276582473}


Now calculate the cooling tower pump power, based of the "Modelling the Influence of Climate on the Performance of the Organic Rankine Cycle for Industrial Waste Heat Recovery" paper.

This assumes a pressure drop aross the cooling tower pump of 100 kPa

In [30]:
# Call function to get Q_cond
Q_cond = orc(T_evap, T_cond)['Q_cond']

# Calculate required heat removal by the trim cooler
T_w_out_evap = orc(T_evap, T_cond)['T_w_out_evap']
T_DC_return = 40 + 273.15  # T of water flowing back into DC
Q_trim = m_w * cpw * (T_w_out_evap - T_DC_return)

# Approximate density of water at condenser temperature
rho_w = 1000 # kg/m^3 (approximate density of water) - should this be mmore specific?

# Calculate volume flow rate of water in condensor loop (not mass flow rate)
V_w_cond = Q_cond/(rho_w*cpw*(T_range))

# Calculate volume flow rate of water in DC water loop - based on established DC water flow rate
V_w_DC = m_w/rho_w

# Define pressure drop in both condenser and DC loop (Pa)
dp = 100e3

# Define pump motor and impeller efficiencies for condenser and DC loop
eta_m, eta_imp = 0.9, 0.78 

# Calculate the work required to pump the water through the condenser loop
W_pump_cond = V_w_cond * dp / (eta_m * eta_imp)
print(f"Condenser cooling tower pump power: {W_pump_cond/1e3:.2f} kW")

# Calculate the work required to pump water through DC loop
W_pump_DC = V_w_DC * dp / (eta_m * eta_imp)
print(f"DC cooling tower pump power: {W_pump_DC/1e3:.2f} kW")


Condenser cooling tower pump power: 20.87 kW
DC cooling tower pump power: 4.09 kW


A wet cooling tower transfers heat by evaporating a small fraction of the circulating water into an air stream. The energy balance requires that the heat rejected by your ORC condenser equals the enthalpy change of the air stream:$$\dot{Q}_{cond} = \dot{m}_{air} \cdot (h_{air,out} - h_{air,in})$$Inlet Air ($h_{air,in}$): Determined by ambient conditions (Dry-bulb temperature $T_{db}$, Wet-bulb temperature $T_{wb}$, and Relative Humidity $RH$).Outlet Air ($h_{air,out}$): Assumed to be saturated ($RH = 100\%$) at an exit temperature close to the warm water inlet temperature (typically estimated using a small approach temperature, e.g., $T_{air,out} = T_{water,in} - 2^\circ\text{C}$).Air Mass Flow Rate ($\dot{m}_{air}$): Calculated directly by rearranging the energy balance:$$\dot{m}_{air} = \frac{\dot{Q}_{cond}}{h_{air,out} - h_{air,in}}$$

In [31]:
from CoolProp.HumidAirProp import HAPropsSI

# --- Psychrometric Cooling Tower Air-Side Balance ---
# 1. Define ambient air conditions at the inlet - assume dry bulb temp.
RH_in = 0.50 # Assume 50% relative humidity for ambient air (check from where)
P_atm = 94100.0 # Atmospheric pressure (Pa) - potentially revise to the specific pressure of las vegas?
T_db_in = 20 + 273.15 # Dry bulb temperature (replace with dry bulb temperature)
RH_out = 1.0 # Saturated air leaving the wet deck - assume 100% saturated

dp_air = 120.0          # Pressure drop across fill/eliminators (Pa) - find justification for this
eta_fan = 0.65          # Fan efficiency - justify
eta_motor_fan = 0.90    # Motor efficiency - justify

# Calculates mass density of inlet air directly (kg/m^3)
rho_air_cond = PS('D', 'T', T_db_in, 'P', P_atm, 'HEOS::Air')

# Calculate inlet air specific enthalpy (J / kg of dry air)
h_air_in_cond = HAPropsSI('H', 'T', T_db_in, 'R', RH_in, 'P', P_atm)
h_air_in_DC = HAPropsSI('H', 'T', T_db_in, 'R', RH_in, 'P', P_atm)

# 2. Define exit air conditions
# Exit air is typically assumed saturated (100% RH) and warmed close to the incoming water temp
T_w_WT_cond = T_wb_amb + T_approach + T_range # Hot water T leaving condensor
T_air_out_cond = T_w_WT_cond - 2.0 # Exit air temperature approaches water-in temperature (find justification for this 2K)

T_air_out_DC = T_w_out_evap - 2.0 # Exit air temperature approaches water-in temperature (find justification for this 2K)


# Calculate exit air specific enthalpy (J / kg of dry air)
h_air_out_cond = HAPropsSI('H', 'T', T_air_out_cond, 'R', RH_out, 'P', P_atm)
h_air_out_DC = HAPropsSI('H', 'T', T_air_out_DC, 'R', RH_out, 'P', P_atm)

# 3. Solve for required air mass flow rate (m_air)
# Q_cond is in Watts
m_air_cond = Q_cond / (h_air_out_cond - h_air_in_cond)
print(f"Condenser Air Mass Flow Rate: {m_air:.3f} kg/s")

m_air_DC = Q_trim / (h_air_out_DC - h_air_in_DC)
print(f"DC Air Mass Flow Rate: {m_air:.3f} kg/s")

# Calculate volume flow rate
V_air_cond = m_air_cond / rho_air # Volume flow rate (m^3/s)
V_air_DC = m_air_DC / rho_air # Volume flow rate (m^3/s)

W_ct_fan_cond = (V_air_cond * dp_air) / (eta_fan * eta_motor_fan)
print(f"Condenser cooling tower fan power: {W_ct_fan_cond/1e3:.2f} kW")

W_ct_fan_DC = (V_air_DC * dp_air) / (eta_fan * eta_motor_fan)
print(f"DC cooling tower fan power: {W_ct_fan_DC/1e3:.2f} kW")

Condenser Air Mass Flow Rate: 95.142 kg/s
DC Air Mass Flow Rate: 95.142 kg/s
Condenser cooling tower fan power: 54.61 kW
DC cooling tower fan power: -1.68 kW


I now calculate the Carnot efficiency of the same system. Carnot efficiency = $\eta_{max} = 1-\frac{T_C}{T_H}$.

I first take $T_C$ as the condensation temperature and $T_H$ as the evaporation temperature, giving the carnot efficiency of the vapour power cycle. This takes into account the fact of pinch points and non-infinite heat exchangers and can be compared to the ORC efficiency above.

Second, I also calculate a Carnot efficiency from the broad reservoir limit. This would be the Carnot efficiency if we assume infinite heat exchangers and therefore provides the ultimate upper cealing for efficiency. $T_C$ is the sink temperature which is the wet-bulb temperature, and $T_H$ is the temperature of the water from the DC (60 degrees)

In [33]:
# Calculating Carnot efficiency of the OCR loop
eta_Carnot_ORC = (1 - T_cond/T_evap)*100
print(f"Carnot efficiency of the ORC loop: {eta_Carnot_ORC:.2f}%")

# Calculating Carnot efficiency from the broad reservoir limit
eta_Carnot_broad = (1 - T_wb_amb/T_DC_out)*100
print(f"Carnot efficiency from the broad reservoir limit: {eta_Carnot_broad:.2f}%")

Carnot efficiency of the ORC loop: 1.98%
Carnot efficiency from the broad reservoir limit: 15.01%
